In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from src.clean import cleanEiaData
from src.fetch_eia import fetchEiaData


def plot_region(g, region, name=None, saveFig=False):

    fig, (ax_top, ax_err) = plt.subplots(
        2, 1, figsize=(10, 5), sharex=True, height_ratios=[3, 1]
    )

    ax_top.plot(g["period"], g["demand"], label="demand")
    ax_top.plot(g["period"], g["forecast"], label="forecast")
    ax_top.set_ylabel("MWh")
    ax_top.set_title(f"{region}: {name}" if name else region)
    ax_top.legend()

    err = g["forecast-error-pct"]
    ax_err.bar(
        g["period"],
        err,
        width=0.8,
        color=["tab:red" if e > 0 else "tab:blue" for e in err],
    )
    ax_err.axhline(0, color="gray", lw=0.8)
    ax_err.set_ylabel("error %")

    fig.tight_layout()

    if saveFig:
        out = Path("reports/figures")
        out.mkdir(parents=True, exist_ok=True)

        fig.savefig(out / f"{region}_forecast_error.png", dpi=150, bbox_inches="tight")

    return fig


REGIONS = {
    "CAL",
    "CAR",
    "CENT",
    "FLA",
    "MIDA",
    "MIDW",
    "NE",
    "NY",
    "NW",
    "SE",
    "SW",
    "TEN",
    "TEX",
}

df, respondents = cleanEiaData(fetchEiaData(start="2019-01-01", end="2019-01-31"))

df = df[df["timezone"] == "Mountain"]
df["level"] = df["respondent"].map(
    lambda r: "us48" if r == "US48" else "region" if r in REGIONS else "ba"
)
df = df[df["level"] == "region"]

wide = (
    df.pivot(index=["period", "respondent"], columns="type", values="value")
    .rename(columns={"D": "demand", "DF": "forecast"})
    .reset_index()
)
wide.columns.name = None

wide["forecast-error-pct"] = (
    (wide["forecast"] - wide["demand"]) / wide["demand"]
) * 100

for region, g in wide.groupby("respondent"):
    plot_region(g, region, respondents[region], True)